In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [43]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
# project_list = ['bids-apps_mrtrix3_connectome', 'leakec_tfc', 'voutcn_megahit', 'moble_quaternion', 'santandermetgroup_downscaler', 'rajeshrinet_pyross', 'google_jax-md', 'ncar_wrf-python', 'juliaintervals_intervalarithmetic.jl', 'magazino_move_base_flex']
projects = ['magazino_move_base_flex']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0037fd55f7da9f2338e1a7f521e469941accce77,magazino_move_base_flex


In [44]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  0%|          | 0/133 [00:00<?, ?it/s]

 71%|███████▏  | 95/133 [01:39<00:39,  1.05s/it]

Got Errors {'b9173c1a7e3588818d0e3671663231c60f6f5b8a': 'Key b9173c1a7e3588818d0e3671663231c60f6f5b8a not found in /da5_fast/All.sha1c/commit_57.tch'}


100%|██████████| 133/133 [02:20<00:00,  1.05s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0037fd55f7da9f2338e1a7f521e469941accce77,c389450b4ba9eea1c2d268d5e228b241e49bedfe,[a9a8860531529cb2a9a06885faea0c9f0576f060],Matthias Holoch <mholoch@gmail.com>,1702311799,+0100,Matthias Holoch <mholoch@gmail.com>,1702311799,+0100,replace boost mutex by std mutex\n,0037fd55f7da9f2338e1a7f521e469941accce77,magazino_move_base_flex
1,0064cf71867825cfdec7cd9607eca8ae477b93d7,a79ec8ae526108d53ac07aee06ea239f86a865d5,[9cff042da8f542352d7af7a584fcb870328a6471],corot <jorge.santos@rapyuta-robotics.com>,1626087704,+0900,corot <jorge.santos@rapyuta-robotics.com>,1626833812,+0900,Allow the controller to handle cancel if prope...,0064cf71867825cfdec7cd9607eca8ae477b93d7,magazino_move_base_flex


In [45]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0037fd55f7da9f2338e1a7f521e469941accce77', 'tree': 'c389450b4ba9eea1c2d268d5e228b241e49bedfe', 'parent': ['a9a8860531529cb2a9a06885faea0c9f0576f060'], 'author': 'Matthias Holoch <mholoch@gmail.com>', 'author_time': 1702311799, 'author_tz': '+0100', 'committer': 'Matthias Holoch <mholoch@gmail.com>', 'committer_time': 1702311799, 'committer_tz': '+0100', 'message': 'replace boost mutex by std mutex\n'}


In [46]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [47]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,magazino_move_base_flex,0037fd55f7da9f2338e1a7f521e469941accce77,Matthias Holoch <mholoch@gmail.com>,1702311799,replace boost mutex by std mutex\n


In [48]:
#mode a means append, so you have all your projects in the same file
yournetid='cliddel2'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## Project Summary

| Name of repo | Number of stars | Number of forks | Last commit date | Number of commits | Number of authors | Max time | Min time |
|---|---:|---:|---|---:|---:|---|---|
| BIDS-Apps/MRtrix3_connectome | 55 | 28 | 04/26/2026 | 473 | 12 | 1773795736 | 1470327983 |
| leakec/tfc | 47 | 13 | 08/01/2026 | 765 | 6 | 1758199663 | 1601671504 |
| voutcn/megahit | 727 | 145 | 10/28/2025 | 738 | 35 | 1761624424 | 1411640958 |
| moble/quaternion | 659 | 92 | 12/15/2025 | 1064 | 45 | 1761608346 | 1310767115 |
| SantanderMetGroup/downscaleR | 112 | 62 | 03/11/2025 | 1141 | 25 | 1742771983 | 1382708378 |
| rajeshrinet/pyross | 167 | 56 | 08/31/2024 | 2269 | 35 | 1725115770 | 1584976412 |
| google/jax-md | 1463 | 249 | 08/18/2026 | 1416 | 83 | 1762560185 | 1557850490 |
| NCAR/wrf-python | 500 | 178 | 08/03/2026 | 844 | 39 | 1776166179 | 1449091912 |
| JuliaIntervals/IntervalArithmetic.jl | 330 | 71 | 09/10/2026 | 3403 | 105 | 1776137504 | 1395244712 |
| magazino/move_base_flex | 549 | 183 | 09/22/2026 | 1329 | 80 | 1757155217 | 1498233087 |
